## Proyecto: Predicción de Spread-F mediante un enfoque multimodal

### Preparación de datos 
En esta sección se leerá y prepará el dataset respectivo. 
Los archivos .hdf5 se encuentran en la carpeta "data_radar"

In [18]:
import h5py as h5
import numpy as np
import matplotlib.pyplot as plt 
import os 
import pandas as pd
import numpy 

from utils import * 
from config import * 

cfg = Config()

### Consideraciones 

Cada archivo .hdf5 contiene datos de radar.
Actualmente la carpeta contiene datos desde 1996 hasta el 2025, donde las mediciones fueron espaciadas contando con pocas mediciones en los primeros años. Sin embargo, desde el 2021 las mediciones fueron realizados de una forma más constante por una nueva configuración del radar. 

En la siguiente celda leeremos los datos de cada archivo. Nos interesa analizar los datos de potencia, que están establecidos por H*W, donde H corresponde a datos en altura y W en tiempo. La resolución en tiempo son de **5 minutos** por cada punto y en altura de **15km**

Todos los datos son guardados en un Dataframe a fin de establecer tiempos y alturas constantes en todos los archivos. Si existe algún bin faltante, se rellena con datos NaN. 

In [24]:
TIME_FREQ = "5min"
HEIGHT_STEP = 15

### Cada 
TIME_TOL = pd.Timedelta("2.5min")
HEIGHT_TOL = HEIGHT_STEP / 2

records = []

for file in sorted(os.listdir(cfg.dir_data_radar)):
 
    try:

        t, h, data = load_file(
            os.path.join(cfg.dir_data_radar, file)
        )

        t = pd.to_datetime(t)

        T, H = np.meshgrid(t, h)

        df_tmp = pd.DataFrame({
            "time": T.ravel(),
            "height": H.ravel(),
            "value": data.ravel()
        })

        records.append(df_tmp)

    except Exception as e:

        print(f"Error en {file}: {e}")

df = pd.concat(records, ignore_index=True)

# Asignar a la grilla deseada
df["time_bin"] = df["time"].dt.round(TIME_FREQ)

df["height_bin"] = (
    np.round(df["height"] / HEIGHT_STEP)
    * HEIGHT_STEP
)

# Calcular errores de asignación
time_error = (
    df["time"] - df["time_bin"]
).abs()

height_error = (
    df["height"] - df["height_bin"]
).abs()

# Mantener sólo puntos suficientemente cercanos
df = df[
    (time_error <= TIME_TOL)
    &
    (height_error <= HEIGHT_TOL)
]

# Si varios puntos caen en la misma celda, promediar
df = (
    df.groupby(
        ["time_bin", "height_bin"],
        as_index=False
    )["value"]
    .mean()
)

df = df.rename(
    columns={
        "time_bin": "time",
        "height_bin": "height"
    }
)

# Crear grilla completa
time_grid = pd.date_range(
    df["time"].min(),
    df["time"].max(),
    freq=TIME_FREQ
)

height_grid = np.arange(
    df["height"].min(),
    df["height"].max() + HEIGHT_STEP,
    HEIGHT_STEP
)

full_grid = pd.MultiIndex.from_product(
    [time_grid, height_grid],
    names=["time", "height"]
)

# Reindexar para que aparezcan los faltantes como NaN
df_final = (
    df.set_index(["time", "height"])
      .reindex(full_grid)
      .reset_index()
)

print(df_final.head())

## guardamos en .parquet
df.to_parquet(os.path.join(cfg.dir_dataset,"data.parquet"))

print("Dataset guardado")


Error en jro20250121drifts.hdf5: "Unable to synchronously open object (object 'range' doesn't exist)"
Error en jro20250122drifts.hdf5: "Unable to synchronously open object (object 'range' doesn't exist)"
Error en jro20250123drifts.hdf5: "Unable to synchronously open object (object 'range' doesn't exist)"
Error en jro20250124drifts.hdf5: "Unable to synchronously open object (object 'range' doesn't exist)"
Error en jro20250125drifts.hdf5: "Unable to synchronously open object (object 'range' doesn't exist)"
Error en jro20250126drifts.hdf5: "Unable to synchronously open object (object 'range' doesn't exist)"
Error en jro20250127drifts.hdf5: "Unable to synchronously open object (object 'range' doesn't exist)"
Error en jro20250128drifts.hdf5: "Unable to synchronously open object (object 'range' doesn't exist)"
Error en jro20250129drifts.hdf5: "Unable to synchronously open object (object 'range' doesn't exist)"
Error en jro20250130drifts.hdf5: "Unable to synchronously open object (object 'ran